# Create co-occurence graph

Steps applied in this notebook

1. Software mentions curated as "not a software" were removed.
2. Mentions without a "mapped to software" entry (i.e., marked as "not\_disambiguated") were normalized to their own software mention.
3. Software mappings with NaN and None values were removed.
4. Only papers that were classified using the open alex dataset were used (see notebook `01_publication_classification` on how these results were obtained). 
5. Papers in which software did not cooccur were removed
6. A cooccurence graph was generated from the remaining software mentions. 

Described in the paper as follows:

For preprocessing, three steps were applied: (1) software mentions curated as "not a software" were removed; (2) NaN and None values were removed; and (3) rows that were unable to disambiguated (i.e., marked as "not\_disambiguated" for "mapped\_to\_software") were normalized to their own software mention. This is the reason why the 'Number of unique disambiguated mentions' row of the 'valid\_doi' column has greatly increased from 97,662 to 891,113.

The co-occurrence graph was constructed by grouping the CZI data by DOI and aggregating on the "mapped to software" field, producing a per-DOI list of software. These lists were then sorted, and a hashmap was used to track all observed software pairs and their frequencies. Iterating over all papers, every pairwise tuple and its frequency was recorded, and this hashmap was subsequently used to construct the co-occurrence graph. Each software vertex was then annotated with the DOIs in which that software was mentioned, along with the total number of papers. Software vertices that did not connect to any other software were removed.

1. [Notebook Setup](#setup)
2. [Cleaning](#cleaning)
    1. [Software Mapping](#normalize)
    2. [Metrics of the cleaning process](#metrics)
    3. [Apply cleaning](#apply_clean)
    4. [Save doi set](#doi_set)
3. [Create cooccurence graph](#graph)
4. [Save cooccurence graph](#save)
5. [How to Load cooccurence graph](#load)

<a id='setup'></a>

## Notebook setup
required packages and directory specifications
The notebook assumes that the dataset files are stored in a folder `data` that sits as the same level as the `notebooks` directory. The assumed directory structure is the following:

- `notebooks`
-  `data` 
    - `software_mentions`
        - `disambiguated`
            - comm_disambiguated.tsv.gz
    - `open_alex_matches`
        - matches.csv.gz
    - `occurence_graph`
        - metrics.txt
        - graph_joblib.pkl.gz

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import igraph as ig
import gzip
from sklearn.model_selection import train_test_split

pd.set_option('max_colwidth', 1000)

In [3]:
# Obtained from 01_publication_classification
ROOT_DISAMBIGUATED_CLASSIFICATION_DIR = '../data/open_alex_matches/'

# Published Sep 19, 2022; Updated Sep 27, 2022 on Dryad. https://doi.org/10.5061/dryad.6wwpzgn2c
ROOT_DISAMBIGUATED_DIR = '../data/software_mentions/'

ROOT_OCCURENCE_GRAPH_DIR = '../data/occurence_graph/'

ROOT_VECTOR_EMBEDDINGS_DIR = '../data/vector/'
abstract_embedding_file = 'abstract_embeddings.csv.gz'


In [ ]:
matches_df = pd.read_csv(ROOT_DISAMBIGUATED_CLASSIFICATION_DIR + '/matches.csv.gz', engine = 'python', compression="gzip")

In [5]:
data_df = pd.read_csv(ROOT_DISAMBIGUATED_DIR + 'disambiguated/comm_disambiguated.tsv.gz', sep = '\\t', engine = 'python', compression = 'gzip')

<a id='normalize'></a>
## Software mapping

The column `mapped_to_software` contains the disambiguated software mention.
When there was no disambiguated term the row was filled with `not_disambiguated`.
However, there are some occurences `mapped_to_software` being `NaN` or `None`.
Additionally, there have some manual annotations been performed that indicate whether the mention is indeed a software in the `curated_label` column. 

In [6]:
data_df = data_df[data_df['mapped_to_software'] != 'not_disambiguated']

<a id='metrics'> </a>
## Metrics of the cleaning process

In [ ]:


ROOT_OCCURRENCE_GRAPH = Path("../data/occurence_graph/")
ROOT_VECTOR_EMBEDDINGS_DIR = Path("../data/vector/")
abstract_embedding_file = "abstract_embeddings.csv.gz"


class MetricsWriter:
    """
    Computes and writes dataset metrics to a text file.
    After write has the 80% papers train embedding, grouped in agg_df 
    After write has the 20% papers test embedding, grouped in agg_test_df
    """
    def __init__(
        self,
        df: pd.DataFrame,
        output_file: str,
        doi_source_df: pd.DataFrame = None,
        agg_output_section_title: str = "",
    ):
        self.df = df.copy()
        self.output_path = ROOT_OCCURRENCE_GRAPH / output_file
        self.doi_source_df = doi_source_df
        self.agg_output_section_title = agg_output_section_title
        self.agg_df = None             
        self.test_embeddings_df = None  
        self.agg_test_df = None

    @staticmethod
    def _parse_embedding(val):
        if isinstance(val, np.ndarray):
            return val
        return np.fromstring(str(val).strip("[]"), sep=" ")

    @staticmethod
    def _load_embeddings_for_dois(doi_set: set) -> pd.DataFrame:
        """
        use the large embedding file to return only rows whose DOI is in doi_set. 
        This will return a dataframe with columns [doi, abs_embedding] :).
        """
        chunks = []
        chunksize = 10_000

        for chunk in pd.read_csv(
            ROOT_VECTOR_EMBEDDINGS_DIR / abstract_embedding_file,
            engine="python",
            compression="gzip",
            chunksize=chunksize,
            usecols=["doi", "abs_embedding"],
        ):
            filtered = chunk[chunk["doi"].isin(doi_set)]
            if not filtered.empty:
                chunks.append(filtered)

        if not chunks:
            return pd.DataFrame(columns=["doi", "abs_embedding"])

        embed_df = pd.concat(chunks, ignore_index=True)
        embed_df["abs_embedding"] = embed_df["abs_embedding"].apply(
            MetricsWriter._parse_embedding
        )
        return embed_df

    @staticmethod
    def _dedup_preserve_order(series):
        items = [
            x for x in series
            if pd.notna(x) and (not isinstance(x, str) or x.strip() != "")
        ]
        seen = set()
        out = []
        for x in items:
            if x not in seen:
                seen.add(x)
                out.append(x)
        return out

    def _compute_main_metrics(self, df0: pd.DataFrame) -> list[str]:
        items_not_software = len(df0[df0["curation_label"] == "not_software"])
        items_not_software_unique = df0[df0["curation_label"] == "not_software"]["software"].nunique()
        count_mapped_nan = df0["mapped_to_software"].isna().sum()
        row_not_software_and_mapped_nan = len(
            df0[(df0["curation_label"] == "not_software") & (df0["mapped_to_software"].isna())]
        )

        df1 = df0[df0["curation_label"] != "not_software"].copy()
        df1 = df1.dropna(subset=["mapped_to_software"])

        all_paper_doi_set = set(df1["doi"].unique())
        has_doi_mask = df1["doi"].notna()
        data_with_doi = df1[has_doi_mask]
        data_without_doi = df1[~has_doi_mask]

        papers_with_doi_nan_included = len(all_paper_doi_set)
        papers_with_doi_valid = data_with_doi["doi"].nunique()
        mentions_with_doi = len(data_with_doi)
        mentions_without_doi = len(data_without_doi)
        unique_mentions = data_with_doi["software"].nunique()
        unique_mentions_mapping = data_with_doi["mapped_to_software"].nunique()

        return [
            f"Number of mentions labeled as 'not software': {items_not_software}",
            f"Number of unique mentions labeled as 'not software': {items_not_software_unique}",
            f"Number of mapped to software that are Nan values: {count_mapped_nan}",
            f"Number of rows that are both not software and have mapped to has Nan values: {row_not_software_and_mapped_nan}",
            "",
            "Metrics after removing Nan values and items labeled as 'not software':",
            f"Number of papers with at least one mention (doi NaN included): {papers_with_doi_nan_included}",
            f"Number of papers with at least one mention (doi NaN omitted): {papers_with_doi_valid}",
            "",
            f"Number of mentions: {mentions_with_doi}",
            f"Number of mentions without a valid doi value: {mentions_without_doi}",
            "",
            f"Number of unique mentions (not disambiguated): {unique_mentions}",
            f"Number of unique mentions (disambiguated mapping): {unique_mentions_mapping}",
        ]
    
    def _build_agg_df(self, df0: pd.DataFrame) -> pd.DataFrame:
        df0 = df0[df0["curation_label"] != "not_software"].copy()
        df0 = df0.dropna(subset=["mapped_to_software"])

        # We filter on the allowed dOI
        if self.doi_source_df is not None and "doi" in self.doi_source_df.columns:
            dois_set = set(self.doi_source_df["doi"])
        else:
            dois_set = set(df0["doi"])

        filtered_df = df0[df0["doi"].isin(dois_set)].copy()

        # Groupby DOi to make it easier to work with 
        agg_df = (
            filtered_df
            .dropna(subset=["doi"])
            .groupby("doi", sort=True, as_index=False)
            .agg(software_list=("mapped_to_software", self._dedup_preserve_order))
        )

        # We save the before image for the metrics
        self._total_papers_before_cooccurrence_filter = len(agg_df)
        self._unique_mentions_before_cooccurrence_filter = len(
            set(x for lst in agg_df["software_list"] for x in (lst or []))
        )

        # The papers are removed with < 2 software before embedding lookup
        agg_df = agg_df[
            agg_df["software_list"].apply(lambda x: len(x) if isinstance(x, list) else 0) >= 2
        ].reset_index(drop=True)

        self._dropped_single_software = self._total_papers_before_cooccurrence_filter - len(agg_df)

        # The papers that have abstracts can continue
        candidate_dois = set(agg_df["doi"].unique())
        embed_df = self._load_embeddings_for_dois(candidate_dois)

        # Papers without embedding are removed.
        dois_with_embedding = set(embed_df["doi"].unique())
        self._dropped_no_embedding = len(candidate_dois - dois_with_embedding)
        agg_df = agg_df[agg_df["doi"].isin(dois_with_embedding)].reset_index(drop=True)

        # We split the list of paper 80 / 20
        all_dois = list(dois_with_embedding)
        dois_80, dois_20 = train_test_split(all_dois, test_size=0.2, random_state=42)

        dois_80_set = set(dois_80)
        dois_20_set = set(dois_20)

        self.train_embeddings_df = (
            embed_df[embed_df["doi"].isin(dois_80_set)]
            .reset_index(drop=True)
        )
        self.test_embeddings_df = (
            embed_df[embed_df["doi"].isin(dois_20_set)]
            .reset_index(drop=True)
        )

        self.agg_test_df = agg_df[agg_df['doi'].isin(dois_20_set)].reset_index(drop=True)
        agg_df = agg_df[agg_df["doi"].isin(dois_80_set)].reset_index(drop=True)

        print(
            f"Embedding split: 80% (community): {len(dois_80)} papers | "
            f"20% (test): {len(dois_20)} papers | "
            f"No embedding (dropped): {self._dropped_no_embedding}"
        )

        return agg_df

    def _compute_agg_metrics(self, agg_df: pd.DataFrame) -> list[str]:
        doi_level_rows = len(agg_df)
        mentions_with_doi_agg = agg_df["doi"].notna().sum()
        mentions_without_doi_agg = agg_df["doi"].isna().sum()
        unique_mentions_agg = len(
            set(x for lst in agg_df["software_list"] for x in (lst or []))
        )

        return [
            self.agg_output_section_title,
            f"Number of paperss with at least one mention: {self._total_papers_before_cooccurrence_filter}",
            f"Number of unique mentions across all papers: {self._unique_mentions_before_cooccurrence_filter}",
            "",
            "filtering papers with less than 2 software mentions (required for co-occurrence graph):",
            f"  Papers dropped (single software only): {self._dropped_single_software}",
            f"  Papers dropped (no embedding found)  : {self._dropped_no_embedding}",
            f"  Papers remaining                     : {doi_level_rows}",
            f"  Papers with a valid doi              : {mentions_with_doi_agg}",
            f"  Papers without a valid doi           : {mentions_without_doi_agg}",
            "",
            "80 / 20 split (on papers with co-occurring software and embeddings):",
            f"  80% community (train) papers: {len(self.train_embeddings_df)}",
            f"  20% held-out  (test)  papers: {len(self.test_embeddings_df)}",
            "",
            f"Number of unique software mentions in co-occurrence papers: {unique_mentions_agg}",
        ]


    def write(self) -> None:
        df0 = self.df.copy()

        main_lines = self._compute_main_metrics(df0)
        agg_df = self._build_agg_df(df0)
        agg_lines = self._compute_agg_metrics(agg_df)

        self.agg_df = agg_df

        lines = main_lines + [""] + agg_lines

        self.output_path.parent.mkdir(parents=True, exist_ok=True)
        self.output_path.write_text("\n".join(lines), encoding="utf-8")
        print(f"Wrote metrics to: {self.output_path}")

        # we save the four outputs.
        # dataframe grouped by DOI part of the training subset
        self.agg_df.to_csv(
            ROOT_OCCURRENCE_GRAPH / "agg_df.csv.gz",
            index=False,
            compression="gzip"
        )
        # dataframe grouped by DOI part of the testing subset
        self.agg_test_df.to_csv(
            ROOT_VECTOR_EMBEDDINGS_DIR / "agg_test_df.csv.gz",
            index=False,
            compression="gzip"
        )
        # dataframe individual DOI with their embedding of the training subset 
        self.train_embeddings_df.to_csv(
            ROOT_VECTOR_EMBEDDINGS_DIR / "train_embeddings.csv.gz",
            index=False,
            compression="gzip"
        )
        # dataframe individual DOI with their embedding of the testing subset 
        self.test_embeddings_df.to_csv(
            ROOT_VECTOR_EMBEDDINGS_DIR / "test_embeddings.csv.gz",
            index=False,
            compression="gzip"
        )

        print(
            f"Saved agg_df ({len(self.agg_df)} rows)\n"
            f"Saved agg_test_df ({len(self.agg_test_df)} rows)\n"
            f"Saved train_embeddings ({len(self.train_embeddings_df)} rows)\n"
            f"Saved test_embeddings  ({len(self.test_embeddings_df)} rows)"
        )

In [ ]:
writer = MetricsWriter(data_df, "metrics.txt", doi_source_df=matches_df, agg_output_section_title="---- agg_df (grouped by DOI) metrics ----")
writer.write()